In [1]:
import aioboto3
import io
import pandas as pd
import asyncio
import boto3
from IPython.display import HTML

In [2]:
s3 = boto3.resource('s3')
bucket_name = 'housekg-etl'
prediction_prefix = 'silver/prediction_price_fact/'
price_fact_prefix = 'silver/realty_price_fact/'
realties_prefix =  'silver/realty_dim/'

cat_cols = ["serie", "district", "micro_district", "condition", "toilet"]
num_cols = ["year", "ceiling_height"]
target_col = "sqm_price"
bucket = s3.Bucket(bucket_name)

async def get_df_from_file(bucket_name, file_key, dfs):
    async with aioboto3.Session().client('s3') as s3:
        buffer = io.BytesIO()
        await s3.download_fileobj(bucket_name, file_key, buffer)
        buffer.seek(0)
        dfs.append(pd.read_parquet(buffer))

async def get_df_from_s3(prefix):
    parquet_files = [obj.key for obj in bucket.objects.filter(Prefix=prefix) if obj.key.endswith('.parquet')]
    files_count = len(parquet_files)
    dfs = []
    await asyncio.gather(
        *(get_df_from_file(bucket_name, file_key, dfs) for file_key in parquet_files)
    )
    return pd.concat(dfs) if dfs else None

In [3]:
fact_prices_df = await get_df_from_s3(price_fact_prefix)
predicted_prices_df = await get_df_from_s3(prediction_prefix)
realties_df = await get_df_from_s3(realties_prefix)
print(f"Fact prices df size: {fact_prices_df.shape}")
print(f"Predicted prices df size: {predicted_prices_df.shape}")
print(f"Realties df size: {realties_df.shape}")

Fact prices df size: (1518, 5)
Predicted prices df size: (7182, 5)
Realties df size: (1437, 14)


In [4]:
current_predicted_prices_df = predicted_prices_df[predicted_prices_df['is_current'] == True].rename(columns={'sqm_price': 'predicted_sqm_price'})
current_fact_prices_df = fact_prices_df[fact_prices_df['is_current'] == True].rename(columns={'sqm_price': 'fact_sqm_price'})


In [5]:
anal_df = current_fact_prices_df.merge(current_predicted_prices_df, on='slug').merge(realties_df, on='slug')
anal_df['ratio'] = anal_df['fact_sqm_price'] / anal_df['predicted_sqm_price']
anal_df['url'] = anal_df['slug'].apply(lambda x: f'https://house.kg/details/{x}')


In [6]:
# HTML(anal_df[['url', 'square', 'fact_sqm_price', 'predicted_sqm_price', 'ratio']].sort_values('ratio').to_html(escape=False))
anal_df[['url', 'square', 'fact_sqm_price', 'predicted_sqm_price', 'ratio']].sort_values('ratio')

,url,square,fact_sqm_price,predicted_sqm_price,ratio
371,https://house.kg/details/869153667cf0ef935a448...,189,941.798942,1936.866455,0.486249
652,https://house.kg/details/4972417682a07aba3b520...,189,952.380952,1848.868286,0.515116
255,https://house.kg/details/9247140681b37400cf589...,186,908.602151,1737.635498,0.522896
14,https://house.kg/details/848322467bffa983ff834...,414,966.183575,1783.638062,0.541693
335,https://house.kg/details/76082696777e69307d9d2...,140,1028.571429,1822.639526,0.564331
...,...,...,...,...,...
561,https://house.kg/details/624862467a0a7a66a3956...,167,2395.209581,1563.994995,1.531469
1283,https://house.kg/details/926449468286c13ae1a22...,52,1673.076923,1049.645996,1.593944
273,https://house.kg/details/7240297681af497c1e476...,29,1534.482759,926.442993,1.656316
1053,https://house.kg/details/262360167ecaab3102e19...,102,2450.980392,1413.779907,1.733636


In [8]:
predicted_prices_df[predicted_prices_df['slug']=='869153667cf0ef935a448-63979105']

,slug,sqm_price,effective_from,effective_to,is_current
23,869153667cf0ef935a448-63979105,1997.802734,2025-05-23 06:45:27.839643,2025-05-24 06:45:34.704421,False
286,869153667cf0ef935a448-63979105,818.301514,2025-05-21 00:10:32.595911,2025-05-21 06:48:38.053511,False
298,869153667cf0ef935a448-63979105,1925.432129,2025-05-21 06:48:38.053511,2025-05-22 06:46:35.032990,False
655,869153667cf0ef935a448-63979105,1930.306274,2025-05-22 06:46:35.032990,2025-05-23 06:45:27.839643,False
646,869153667cf0ef935a448-63979105,1936.866455,2025-05-24 06:45:34.704421,NaT,True
